In [1]:
# ── Cell 0 · Bootstrap ──────────────────────────────────────────────────────
# Walk up from the notebook until we find the folder containing .env (the repo
# root), then put <root>/src on sys.path so shared modules import on either machine.
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                            if (p / '.env').exists()) / 'src'))

from config import *            # PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, DOWNLOADS_DIR, CURRENT_YEAR, …
from download_log import load_log, update_entry, print_entry, print_stale_sources

import os                       # filesystem paths
import numpy as np              # numeric mapping / NaN handling
import pandas as pd             # all tabular work

log = load_log()                # shared currency log — "data as-of" is derived from data, never hardcoded

## 33 · PEFA — Public Expenditure & Financial Accountability (PFM quality)

**Source:** PEFA "Scores Downloads" (`pefa.org/assessments/batch-downloads`) — structured CSV of A–D indicator/dimension scores. **Acquisition: manual download → Downloads → auto-detect.** Programmatic fetch not implemented (site is a Drupal form POST); flagged as an automation follow-up.

**Scope (see `framework_decisions.md`):** core = **2016 framework, national, latest assessment per country** (~85 countries; median assessment year 2022, all ≤9 yrs old). The **2011 framework is deferred** — its 35 national-only additions are 11–18 yrs stale for the investment-relevant names (Brazil 2009, India 2010, Norway 2008), so current 55% coverage beats stale 77%. `assessment_year` is carried as a recency flag. Raw retains 2011 for a possible narrow, flagged backfill revisited at PFM-concept assembly.

**Numeric convention:** A–D → indicator-level 7-point (D=1, D+=1.5, C=2, C+=2.5, B=3, B+=3.5, A=4); dimension-level 4-point. `*` → letter + quality flag (meaning to verify vs PEFA handbook); `NU`/`NaN` → missing.

**Feeds:** several governance concepts (PI-9 transparency, PI-10 fiscal risk, PI-26/30/31 audit & scrutiny, PI-13 debt, PI-24 procurement, …). Indicator→concept mapping is downstream; this notebook outputs a clean tidy table only.

In [2]:
# ── Cell 2 · Ingest raw PEFA scores ─────────────────────────────────────────
# PEFA is a MANUAL-download source: on the Scores Downloads page select
# Framework=2016, Type=National, Status=Final → Download (lands in ~/Downloads as
# assessments_<unixtime>.csv). This cell auto-detects ALL such files, merges them,
# snapshots a raw copy into RAW_DIR, and reports what was ingested before filtering.
# We intentionally keep ALL frameworks in raw (2011 retained for the deferred,
# possible narrow backfill); the 2016-national filter happens in the next cell.
import glob  
# not in the bootstrap; needed here
# ── ⚙️ MANUAL PARAMETER (documented in instructions_data_maintenance.md) ──────
# PEFA framework version this pipeline scores. Deliberate SCOPE choice (2016 core;
# 2011 deferred as stale) — cannot be auto-derived, so it is the single knob to
# change if/when PEFA releases a new framework version.
PEFA_FRAMEWORK = "2016"

RAW_PEFA = os.path.join(RAW_DIR, "pefa_assessments_raw.csv")      # canonical raw snapshot (gitignored)

# 1) Prefer fresh files from Downloads; fall back to the existing raw snapshot.
dl_files = sorted(glob.glob(os.path.join(DOWNLOADS_DIR, "assessments_*.csv")),
                  key=os.path.getmtime)
if dl_files:
    raw = (pd.concat([pd.read_csv(f) for f in dl_files], ignore_index=True)
             .drop_duplicates())                                  # merge + drop exact-dup rows
    raw.to_csv(RAW_PEFA, index=False)                             # snapshot to RAW_DIR (replaces prior)
    src = f"{len(dl_files)} file(s) from Downloads -> snapshot"
elif os.path.exists(RAW_PEFA):
    raw = pd.read_csv(RAW_PEFA)                                   # no new download; reuse snapshot
    src = "existing RAW_DIR snapshot (no new download found)"
else:
    raise FileNotFoundError(
        "No assessments_*.csv in Downloads and no raw snapshot. Download the PEFA "
        "2016/National/Final set from pefa.org/assessments/batch-downloads into Downloads.")

raw['Framework'] = raw['Framework'].astype(str)                  # normalize framework label dtype

# 2) Report what we ingested (all frameworks, pre-filter) so we can verify.
print("ingested from:", src)
print("raw shape:", raw.shape)
print("\nframework composition:")
print(raw['Framework'].value_counts(dropna=False).to_string())
print("\nyear range per framework:")
print(raw.groupby('Framework')['Year'].agg(['min', 'max', 'count']).to_string())

ingested from: 3 file(s) from Downloads -> snapshot
raw shape: (378, 143)

framework composition:
Framework
2011    201
2016    177

year range per framework:
            min   max  count
Framework                   
2011       2005  2016    201
2016       2016  2026    177


In [3]:
# ── Cell 3 · Filter to the core: 2016 framework, national, latest per country ─
# Scope (framework_decisions.md): core = 2016 framework, national only, latest
# assessment per country. Subnational entities use "Country - Subentity" naming
# (e.g. "Georgia - Tbilisi", "Kenya - Makueni County"); we exclude them. 2011
# stays in raw only (deferred backfill). No hardcoded years anywhere here.

# 1) keep 2016 framework
core = raw[raw['Framework'] == PEFA_FRAMEWORK].copy()   # scope param from Cell 2 (no hardcoded version)

# 2) drop subnational rows. The reliable marker is " - " (space-hyphen-space);
#    descriptor words are belt-and-suspenders. Printed below so we can verify no
#    genuine national country is wrongly dropped.
SUBNAT = r" - |Municipalit|Provincial|Province|County|Region of|State of|District|Canton|Prefecture|Department of"
is_subnat = core['Country'].str.contains(SUBNAT, case=False, na=False, regex=True)
print("excluded subnational rows:", int(is_subnat.sum()))
print(sorted(core.loc[is_subnat, 'Country'].unique()))
core = core[~is_subnat]

# 3) dedup to the latest assessment per country (max Year wins)
core = core.sort_values('Year').drop_duplicates('Country', keep='last').reset_index(drop=True)

# 4) report
print(f"\ncore national 2016 countries: {core['Country'].nunique()}  (rows: {len(core)})")
print("assessment-year distribution:")
print(core['Year'].astype(int).describe()[['min', '25%', '50%', '75%', 'max']].to_string())
print("\nsample (country, year):")
print(core[['Country', 'Year']].sort_values('Country').head(20).to_string(index=False))

excluded subnational rows: 59
['Albania - Berat Municipality', 'Albania - Fier Municipality', 'Albania - Tirana Municipality', 'Albania - Tropoja Municipality', 'Bolivia - La Paz', 'Bosnia and Herzegovina - BiH', 'Bosnia and Herzegovina - District Brčko', 'Bosnia and Herzegovina - FBiH', 'Bosnia and Herzegovina - Republika Srpska', 'Cameroon - Ville de Douala', 'Ethiopia - Addis Ababa City', 'Ethiopia - Amhara Region', 'Ethiopia - Oromia Region', 'Ethiopia - Somali Region', 'Ethiopia - Southern Nations & Nationalities Peoples’ Region', 'Ethiopia - Tigray Region', 'Georgia - Batumi', 'Georgia - Martvili', 'Georgia - Tbilisi', 'Germany - Germany - Bad Laer Municipality', 'Germany - Germany-Osnabrueck City', 'Germany - Germany-Osnabrueck District', 'Germany - Germany-Westerkappeln Municipality', 'Jordan - Greater Amman Municipality', 'Kenya - Baringo County', 'Kenya - Kajiado County', 'Kenya - Kakamega County', 'Kenya - Kenya Nakuru County', 'Kenya - Makueni County', 'Kenya - West Pokot C

In [4]:
# ── Cell 4 · Harmonize country names → ISO3 (canonical join key) ─────────────
# House pattern (same as AREAER/FARI etc.): map source names → ISO3 via pycountry,
# with a manual OVERRIDES dict for labels pycountry can't resolve. PEFA ships names
# only; the framework keys every source on ISO3 (country_code).
import pycountry

# Manual overrides for PEFA labels pycountry misses (extend as the report flags more).
OVERRIDES = {
    "Cote d'Ivoire": "CIV", "Kyrgyz Republic": "KGZ", "Lao PDR": "LAO",
    "West Bank and Gaza": "PSE", "Cabo Verde": "CPV", "Cook Islands": "COK",
    "Democratic Republic of Congo": "COD", "Republic of Congo": "COG",
    "Gambia, The": "GMB", "The Gambia": "GMB", "Micronesia": "FSM",
    "Sao Tome and Principe": "STP", "Montseratt": "MSR", "Kosovo": "XKX",
    "Eswatini": "SWZ", "Timor-Leste": "TLS",
    "Macedonia": "MKD", "St. Pierre and Miquelon": "SPM",
}

def to_iso3(name):
    if name in OVERRIDES:                                  # manual map first
        return OVERRIDES[name]
    try:
        return pycountry.countries.lookup(name).alpha_3    # pycountry resolves the rest
    except LookupError:
        return None                                        # unmatched -> flagged below

core['country_code'] = core['Country'].map(to_iso3)

# report unmapped (to extend OVERRIDES) and guard against ISO3 collisions
unmapped = sorted(core.loc[core['country_code'].isna(), 'Country'].unique())
print(f"countries: {len(core)} | mapped: {core['country_code'].notna().sum()} | unmapped: {len(unmapped)}")
print("UNMAPPED (add to OVERRIDES):", unmapped if unmapped else "none ✓")
dups = core['country_code'].value_counts()
print("duplicate ISO3 (should be none):", dups[dups > 1].to_dict())

countries: 85 | mapped: 85 | unmapped: 0
UNMAPPED (add to OVERRIDES): none ✓
duplicate ISO3 (should be none): {}


In [5]:
# ── Cell 5 · Reshape wide→long + numeric mapping + level/quality flags ───────
# Wide PEFA scores (one col per indicator PI-XX and dimension PI-XX.Y) -> tidy long.
# Convention: A–D -> numeric, 7-point at indicator level (D=1, D+=1.5 … A=4), 4-point
# at dimension level (dims never carry '+'). '*' is a qualifier on a valid grade (A*
# exists => not "unscorable") -> strip but record quality_flag. 'NU'/blank -> missing.

ID_COLS = ['country_code', 'Country', 'Year', 'Framework']

# 1) drop all-NaN columns (2011-framework-only columns, empty for 2016 rows)
core_2016 = core.dropna(axis=1, how='all')
score_cols = [c for c in core_2016.columns if c not in ID_COLS]

# 2) wide -> long
long = core_2016.melt(id_vars=ID_COLS, value_vars=score_cols,
                      var_name='indicator_code', value_name='raw_grade')

# 3) level: dimension codes contain '.', indicators do not
long['level'] = np.where(long['indicator_code'].str.contains(r'\.'), 'dimension', 'indicator')

# 4) clean grade text; extract '*' qualifier; normalize missing tokens
g = long['raw_grade'].astype(str).str.strip().str.upper()
long['quality_flag'] = g.str.contains(r'\*', na=False)
g = g.str.replace('*', '', regex=False).replace({'NU': np.nan, 'NAN': np.nan, 'NR': np.nan, '': np.nan})

# 5) grade -> numeric (one dict; dimensions simply never use the '+' keys)
GRADE_NUM = {'D': 1.0, 'D+': 1.5, 'C': 2.0, 'C+': 2.5, 'B': 3.0, 'B+': 3.5, 'A': 4.0}
long['numeric_score'] = g.map(GRADE_NUM)

# 6) validation: any grade token we failed to map?
unmapped_grades = sorted(set(g.dropna()) - set(GRADE_NUM))

# 7) keep rows with a real grade; final tidy schema
before = len(long)
long = long.dropna(subset=['numeric_score'])
long = long.rename(columns={'Country': 'country_name_source',
                            'Year': 'assessment_year',
                            'Framework': 'framework_version'})
long = long[['country_code', 'country_name_source', 'assessment_year', 'framework_version',
             'indicator_code', 'level', 'raw_grade', 'quality_flag', 'numeric_score']
            ].reset_index(drop=True)

# 8) report
print(f"long rows: {before} -> {len(long)} (dropped {before-len(long)} NU/blank/unscored)")
print("unmapped grade tokens (MUST be empty):", unmapped_grades)
print("level counts:", long['level'].value_counts().to_dict())
print("asterisk-flagged rows:", int(long['quality_flag'].sum()))
print("\nnumeric_score by level:")
print(long.groupby('level')['numeric_score'].describe()[['count', 'mean', 'min', 'max']].to_string())
print("\nsample:")
print(long.head(8).to_string(index=False))

long rows: 10965 -> 10380 (dropped 585 NU/blank/unscored)
unmapped grade tokens (MUST be empty): []
level counts: {'dimension': 7759, 'indicator': 2621}
asterisk-flagged rows: 204

numeric_score by level:
            count      mean  min  max
level                                
dimension  7759.0  2.418224  1.0  4.0
indicator  2621.0  2.346433  1.0  4.0

sample:
country_code country_name_source  assessment_year framework_version indicator_code     level raw_grade  quality_flag  numeric_score
         IRQ                Iraq             2017              2016          PI-01 indicator         D         False            1.0
         BFA        Burkina Faso             2017              2016          PI-01 indicator         D         False            1.0
         GAB               Gabon             2017              2016          PI-01 indicator         D         False            1.0
         SYC          Seychelles             2017              2016          PI-01 indicator         A    

In [6]:
# ── Cell 6 · Validation gates + currency + save processed ────────────────────
# Gate the clean long table before persisting, derive currency from the data
# (no hardcoded years/counts), then write pefa_clean.csv to PROCESSED_DIR.

# 1) GATE — one row per (country, indicator_code): no duplicates
dup = long.duplicated(subset=['country_code', 'indicator_code'])
assert not dup.any(), f"duplicate country×indicator rows: {int(dup.sum())}"

# 2) GATE — numeric score within the A–D band [1, 4]
assert long['numeric_score'].between(1, 4).all(), "numeric_score outside [1,4]"

# 3) GATE — sanity floor on country count (NOT a hardcoded exact count; refresh-safe)
n_ctry = long['country_code'].nunique()
assert n_ctry >= 50, f"suspiciously few countries ({n_ctry}) — check the upstream filter"

# 4) coverage diagnostic — indicator-level completeness per country
ind = long[long['level'] == 'indicator']
n_ind_total = ind['indicator_code'].nunique()                 # distinct PIs observed (derived, not hardcoded)
cov = ind.groupby('country_code')['indicator_code'].nunique()
print(f"distinct countries: {n_ctry} | distinct indicators observed: {n_ind_total}")
print(f"\nindicators present per country (of {n_ind_total}):")
print(cov.describe()[['min', '25%', '50%', '75%', 'max']].to_string())
sparse = cov[cov < 0.8 * n_ind_total]
print("sparse countries (<80% of indicators):", sparse.to_dict() or "none")

# 5) currency — derived from the data, never hardcoded
data_as_of = int(long['assessment_year'].max())
print(f"\nassessment_year span: {int(long['assessment_year'].min())}–{data_as_of} "
      f"| data_as_of = {data_as_of}")

# 6) SAVE — tidy long table to PROCESSED_DIR (git-tracked), replacing any prior
OUT = os.path.join(PROCESSED_DIR, "pefa_clean.csv")
long.to_csv(OUT, index=False)
print("\nwrote:", OUT, "| shape:", long.shape)

distinct countries: 85 | distinct indicators observed: 31

indicators present per country (of 31):
min    29.0
25%    31.0
50%    31.0
75%    31.0
max    31.0
sparse countries (<80% of indicators): none

assessment_year span: 2017–2026 | data_as_of = 2026

wrote: C:\Users\mjbou\governance-framework\data\processed\pefa_clean.csv | shape: (10380, 9)


In [7]:
# ── Cell 7 · Update the shared download log ──────────────────────────────────
# Record PEFA's currency. Both dates are DERIVED — data_as_of from the data,
# download date from today's run.
from datetime import date

update_entry(
    "PEFA",
    last_successful_download_date=str(date.today()),          # manual download captured today
    data_as_of_date=str(data_as_of),                          # newest assessment year (derived in Cell 6)
    local_filename="pefa_assessments_raw.csv",                # raw snapshot in RAW_DIR
    latest_available_version=f"{'/'.join(sorted(long['framework_version'].unique()))} Framework",  # derived
    notes=("National, latest assessment per country "
           f"({n_ctry} countries, {n_ind_total} indicators, "
           f"span {int(long['assessment_year'].min())}-{data_as_of}). "
           "Manual download from pefa.org Scores Downloads. Framework scope & "
           "backfill rationale in framework_decisions.md."),
)
print_entry("PEFA")                                           # confirm the logged row

[download_log] Updated entry for PEFA
  source_id: PEFA
  last_attempted_date: 2026-06-18
  last_successful_download_date: 2026-06-18
  data_as_of_date: 2026
  local_filename: pefa_assessments_raw.csv
  latest_available_version: 2016 Framework
  no_update_reason: nan
  notes: National, latest assessment per country (85 countries, 31 indicators, span 2017-2026). Manual download from pefa.org Scores Downloads. Framework scope & backfill rationale in framework_decisions.md.
